# 💰 Web Scraping — Saldo y Programación Financiera del F12B

**Fuente:** https://ofi5.mef.gob.pe/inviertews/Repseguim/ResumF12B?codigo={CUI}

Extrae por CUI:
- **Programación Financiera Actualizada**
- **Déficit o Saldo**

---
### 📋 Instrucciones
1. Coloca tu archivo `CUI.xlsx` en la **misma carpeta** que este notebook
2. El archivo debe tener una columna llamada **`CUI`** (o `CODIGO`)
3. Edita solo la celda de **Configuración** (PASO 2)
4. Ejecuta las celdas en orden con **Shift+Enter**
5. El progreso se guarda automáticamente

### ⚙️ PASO 1 — Instalación de dependencias
> Ejecuta solo la primera vez o en un entorno nuevo.

In [ ]:
!pip install selenium webdriver-manager openpyxl pandas

### ⚙️ PASO 2 — Configuración
> **⚠️ Esta es la ÚNICA celda que debes editar.**

In [ ]:
from pathlib import Path

# ================================================================
# CONFIGURACIÓN — Solo edita esta sección
# ================================================================

# Carpeta base: detecta automáticamente dónde está el notebook
CARPETA_BASE = Path.cwd()

# --- Archivos ---
NOMBRE_ARCHIVO_ENTRADA = "CUI.xlsx"                              # <- Cambia si tu archivo tiene otro nombre
NOMBRE_ARCHIVO_SALIDA  = "resultados_saldo_programacion.xlsx"    # <- Nombre del archivo de resultados

RUTA_ENTRADA = CARPETA_BASE / NOMBRE_ARCHIVO_ENTRADA
RUTA_SALIDA  = CARPETA_BASE / NOMBRE_ARCHIVO_SALIDA

# --- Parámetros del scraping ---
MAX_REINTENTOS    = 2     # Reintentos por CUI fallido
MODO_VISIBLE      = True  # True = Chrome visible | False = silencioso
GUARDAR_CADA      = 5     # Guardar progreso cada N CUIs
ESPERA_ENTRE_CUIS = (1.5, 3.0)  # Espera aleatoria anti-saturación (seg)

# --- Timeouts en segundos ---
TIMEOUT_PAGINA   = 20
TIMEOUT_ELEMENTO = 10

# ================================================================
# Verificación automática
# ================================================================
print(f"📁 Carpeta de trabajo : {CARPETA_BASE}")
print(f"📥 Archivo de entrada : {NOMBRE_ARCHIVO_ENTRADA}")
print(f"📤 Archivo de salida  : {NOMBRE_ARCHIVO_SALIDA}")
print()
if RUTA_ENTRADA.exists():
    print(f"✅ '{NOMBRE_ARCHIVO_ENTRADA}' encontrado — listo para continuar")
else:
    print(f"❌ ERROR: No se encontró '{NOMBRE_ARCHIVO_ENTRADA}' en:")
    print(f"   {CARPETA_BASE}")
    print("   Coloca el archivo en esa carpeta y vuelve a ejecutar esta celda.")

### ⚙️ PASO 3 — Importar librerías

In [ ]:
import pandas as pd
import time
import random
from selenium import webdriver
from selenium.webdriver import Chrome
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait as Wait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from webdriver_manager.chrome import ChromeDriverManager

print("✅ Librerías importadas correctamente")

### ⚙️ PASO 4 — Funciones de checkpoint (guardar y reanudar progreso)

In [ ]:
def cargar_progreso():
    """Carga el progreso previo si existe el archivo de salida."""
    if RUTA_SALIDA.exists():
        try:
            df = pd.read_excel(RUTA_SALIDA)
            print(f"📥 Progreso encontrado: {len(df)} CUIs ya procesados")
            return df.to_dict('records')
        except Exception as e:
            print(f"⚠️ No se pudo leer el progreso anterior: {e}")
    return []

def obtener_pendientes(completa, procesados):
    """Calcula los CUIs pendientes."""
    if not procesados:
        print(f"📋 Todos los CUIs pendientes: {len(completa)}")
        return completa
    cuis_ok = {str(r['CUI']) for r in procesados}
    pendientes = [cui for cui in completa if str(cui) not in cuis_ok]
    if pendientes:
        print(f"⏳ Pendientes: {len(pendientes)}")
    return pendientes

def guardar(resultados):
    """Guarda el progreso en el archivo de salida."""
    try:
        pd.DataFrame(resultados).to_excel(RUTA_SALIDA, index=False)
        return True
    except Exception as e:
        print(f"⚠️ Error al guardar: {e}")
        return False

print("✅ Funciones de checkpoint listas")

### ⚙️ PASO 5 — Configurar el navegador Chrome

In [ ]:
def crear_driver():
    """Crea Chrome con configuración optimizada para scraping."""
    service = Service(ChromeDriverManager().install())
    options = webdriver.ChromeOptions()

    if not MODO_VISIBLE:
        options.add_argument("--headless=new")
    else:
        options.add_argument("--start-maximized")

    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-extensions")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-logging")
    options.add_argument("--log-level=3")
    options.add_argument("--silent")

    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.default_content_setting_values.notifications": 2,
        "profile.default_content_setting_values.media_stream": 2,
    }
    options.add_experimental_option("prefs", prefs)
    options.add_experimental_option('excludeSwitches', ['enable-logging'])
    options.page_load_strategy = 'normal'

    driver = Chrome(service=service, options=options)
    driver.set_page_load_timeout(TIMEOUT_PAGINA)
    driver.set_script_timeout(15)
    return driver

print("✅ Función crear_driver() lista")

### ⚙️ PASO 6 — Funciones de extracción (Saldo y Programación Financiera)

In [ ]:
def extraer_saldo_programacion(driver):
    """Extrae Programación Financiera Actualizada y Déficit/Saldo del F12B."""
    try:
        Wait(driver, TIMEOUT_ELEMENTO).until(
            EC.presence_of_element_located((By.ID, "prfin"))
        )
        time.sleep(0.8)

        datos = driver.execute_script("""
            function getText(id) {
                try {
                    var elem = document.getElementById(id);
                    if (!elem) return 'NO DISPONIBLE';
                    return (elem.textContent || elem.innerText || '').trim() || 'NO DISPONIBLE';
                } catch(e) { return 'NO DISPONIBLE'; }
            }
            return {
                programacion_financiera: getText('prfin'),
                deficit_saldo: getText('defsal')
            };
        """)

        if (datos['programacion_financiera'] == 'NO DISPONIBLE' and
                datos['deficit_saldo'] == 'NO DISPONIBLE'):
            return None

        return datos

    except TimeoutException:
        print("⏱️", end=" ")
        return None
    except Exception:
        print("⚠️", end=" ")
        return None

def procesar_cui(driver, cui, intento=1):
    """Navega a la URL del F12B para el CUI y extrae los datos."""
    try:
        url = f"https://ofi5.mef.gob.pe/inviertews/Repseguim/ResumF12B?codigo={cui}"
        driver.get(url)
        time.sleep(0.5)

        datos = extraer_saldo_programacion(driver)
        if not datos:
            raise Exception("No se pudieron extraer los datos")

        return datos

    except Exception as e:
        if intento < MAX_REINTENTOS:
            print(f"🔄{intento+1}", end=" ")
            time.sleep(1.5)
            return procesar_cui(driver, cui, intento + 1)
        else:
            raise e

print("✅ Funciones de extracción listas")

### ▶️ PASO 7 — Cargar CUIs e inicializar el navegador

In [ ]:
print("📂 Cargando CUIs...")
df_cui = pd.read_excel(RUTA_ENTRADA)

# Detectar columna de CUIs (puede llamarse CUI o CODIGO)
if 'CUI' in df_cui.columns:
    lista_completa = df_cui['CUI'].astype(str).tolist()
elif 'CODIGO' in df_cui.columns:
    lista_completa = df_cui['CODIGO'].astype(str).tolist()
else:
    columnas = ', '.join(df_cui.columns.tolist())
    raise ValueError(f"No se encontró columna 'CUI' o 'CODIGO'. Columnas disponibles: {columnas}")

print(f"📊 Total CUIs en archivo: {len(lista_completa)}")

resultados = cargar_progreso()
lista_cuis = obtener_pendientes(lista_completa, resultados)

if not lista_cuis:
    print("🎉 ¡Todos los CUIs ya fueron procesados!")
else:
    driver = crear_driver()
    print(f"\n⚡ Listo para procesar {len(lista_cuis)} CUIs pendientes")

### ▶️ PASO 8 — Ejecutar el scraping
> ⚠️ **No cierres Chrome** mientras se ejecuta.
> Interrumpe con **⏹️ Stop** de Jupyter; el progreso se guarda automáticamente.
> Re-ejecuta los PASOS 7 y 8 para continuar.

In [ ]:
tiempo_inicio = time.time()
exitosos = 0
fallos   = 0

print(f"{'='*65}")
print(f"🚀 INICIO: {time.strftime('%H:%M:%S')}")
print(f"{'='*65}\n")

try:
    for idx, cui in enumerate(lista_cuis, 1):
        t_inicio = time.time()
        print(f"[{idx}/{len(lista_cuis)}] {cui}:", end=" ")

        try:
            datos = procesar_cui(driver, cui)
            resultados.append({
                "CUI": cui,
                "Programación Financiera Actualizada": datos['programacion_financiera'],
                "Déficit o Saldo": datos['deficit_saldo']
            })
            exitosos += 1
            print(f"✅ ({time.time()-t_inicio:.1f}s)")

        except Exception:
            resultados.append({
                "CUI": cui,
                "Programación Financiera Actualizada": "NO DISPONIBLE",
                "Déficit o Saldo": "NO DISPONIBLE"
            })
            fallos += 1
            print(f"❌ ({time.time()-t_inicio:.1f}s)")

        # Guardar periódicamente
        if (idx % GUARDAR_CADA == 0) or (idx == len(lista_cuis)):
            if guardar(resultados) and idx % GUARDAR_CADA == 0:
                print(f"   💾 Guardado")

        # Espera anti-saturación
        if idx < len(lista_cuis):
            time.sleep(random.uniform(ESPERA_ENTRE_CUIS[0], ESPERA_ENTRE_CUIS[1]))

        # Estadísticas cada 10 CUIs
        if idx % 10 == 0:
            t_el  = time.time() - tiempo_inicio
            tasa   = idx / t_el * 60
            eta    = (len(lista_cuis) - idx) * (t_el / idx)
            print(f"   📊 {exitosos}✅ {fallos}❌ | {t_el:.0f}s | ETA: {eta/60:.1f}min | {tasa:.1f} CUI/min\n")

except KeyboardInterrupt:
    print("\n\n⚠️ INTERRUMPIDO — guardando progreso...")
    guardar(resultados)
    driver.quit()
    print(f"💾 Progreso guardado. Re-ejecuta los PASOS 7 y 8 para continuar.")

else:
    driver.quit()
    guardar(resultados)
    t_total = time.time() - tiempo_inicio

    print(f"\n{'='*65}")
    print("🏁 COMPLETADO")
    print(f"{'='*65}")
    print(f"📊 Total procesados: {len(lista_cuis)}")
    print(f"✅ Exitosos : {exitosos}")
    print(f"❌ Fallidos : {fallos}")
    print(f"⏱️  Tiempo  : {t_total/60:.1f} minutos ({t_total:.1f}s)")
    if t_total > 0:
        print(f"⚡ Velocidad: {len(lista_cuis)/t_total*60:.1f} CUI/min")
    print(f"💾 Guardado en: {RUTA_SALIDA}")
    print(f"{'='*65}")

### 📊 PASO 9 — Ver estadísticas del archivo final (opcional)

In [ ]:
if RUTA_SALIDA.exists():
    df_final = pd.read_excel(RUTA_SALIDA)
    col      = 'Programación Financiera Actualizada'
    total       = len(df_final)
    con_datos   = len(df_final[df_final[col] != 'NO DISPONIBLE'])
    sin_datos   = total - con_datos
    print(f"{'='*60}")
    print(f"📊 ESTADÍSTICAS TOTALES")
    print(f"{'='*60}")
    print(f"   Total CUIs : {total}")
    print(f"   ✅ Con datos : {con_datos} ({con_datos/total*100:.1f}%)")
    print(f"   ❌ Sin datos : {sin_datos} ({sin_datos/total*100:.1f}%)")
    print(f"   📁 Archivo  : {RUTA_SALIDA}")
    print(f"{'='*60}")
    print("\n📋 Preview (últimas 5 filas):")
    df_final.tail(5)
else:
    print("⚠️ No se encontró el archivo de resultados. Ejecuta el scraping primero.")